In [ ]:
import pandas as pd
import numpy as np
import os
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

# Konfigurasi path direktori
BASE_DIR = ".."
DATA_PATH = os.path.join(BASE_DIR, "data")
SUB_PATH = os.path.join(BASE_DIR, "submissions")

# Fungsi Downcasting: Mengubah float64 ke float32 & int64 ke int32
def reduce_mem(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        if df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df

print("Cell 1: Setup Selesai!")

Cell 1: Setup Selesai!


In [ ]:
# Memuat dataset mentah
df_train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
df_test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

# Cleaner Battery_Level: Menghapus karakter '%' dan konversi ke numeric
def fix_battery(val):
    if pd.isna(val): return np.nan
    s = str(val).replace('%', '')
    try: return float(s)
    except: return np.nan

df_train['Battery_Level'] = df_train['Battery_Level'].apply(fix_battery)
df_test['Battery_Level'] = df_test['Battery_Level'].apply(fix_battery)

# Eksekusi penghematan memori
df_train = reduce_mem(df_train)
df_test = reduce_mem(df_test)

print("Cell 2: Data dimuat dan dibersihkan!")

Cell 2: Data dimuat dan dibersihkan!


In [ ]:
def build_expert_features(df):
    # Temporal Feature: Ekstrak jam untuk pola kemacetan/kejahatan malam hari
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['hour'] = df['Timestamp'].dt.hour
    
    # Economic Anomaly: Deteksi Fraud (Jarak pendek tapi harga selangit)
    df['price_dist_ratio'] = df['Est_Price_IDR'] / (df['Distance_KM'] + 0.5)
    df['is_suspicious_short'] = ((df['Distance_KM'] < 1.0) & (df['Est_Price_IDR'] > 30000)).astype(int)
    
    # Physical Dynamics: Magnitude sensor untuk deteksi gaya mengemudi (Safety Violation)
    df['accel_mag'] = np.sqrt(df['Accel_X']**2 + df['Accel_Y']**2 + df['Accel_Z']**2)
    df['gyro_abs'] = df['Gyro_Z'].abs()
    
    # Hardware Categorization: Ambil merk perangkat (Device Profiling)
    df['Device_Brand'] = df['Device_FP'].astype(str).apply(lambda x: x.split('-')[0])
    
    # Spatial Delta: Hitung jarak lurus (Euclidean) antara pickup dan dropoff
    df['lat_diff'] = (df['Dropoff_Lat'] - df['Pickup_Lat']).abs()
    df['long_diff'] = (df['Dropoff_Long'] - df['Pickup_Long']).abs()
    df['euclidean_dist'] = np.sqrt(df['lat_diff']**2 + df['long_diff']**2)
    
    # Route Efficiency: Rasio jarak tempuh vs jarak lurus (Navigation Issue detector)
    df['route_efficiency'] = df['Distance_KM'] / (df['euclidean_dist'] + 0.001)

    # Drop kolom yang tidak berguna untuk modeling (ID dan Timestamp mentah)
    drop_cols = ['Timestamp', 'Trip_ID'] if 'Trip_ID' in df.columns else ['Timestamp']
    return df.drop(columns=drop_cols)

# Transformasi fitur untuk train dan test
y = df_train['Trip_Label']
X = build_expert_features(df_train.drop(columns=['Trip_Label']))
X_test_final = build_expert_features(df_test)

print("Cell 3: Feature Engineering Berhasil!")

Cell 3: Feature Engineering Berhasil!


In [ ]:
# Sampling 15% dari total data untuk menjaga stabilitas RAM selama training
X_samp, _, y_samp, _ = train_test_split(X, y, train_size=0.15, stratify=y, random_state=42)

# Strategi Oversampling: Duplikasi manual kelas minoritas (Navigation & Service)
minoritas_mask = y_samp.isin(['Navigation_Issue', 'Service_Complaint'])
X_min = X_samp[minoritas_mask]
y_min = y_samp[minoritas_mask]

# Penggabungan: Data original + 2x duplikasi minoritas
X_oversampled = pd.concat([X_samp, X_min, X_min], axis=0)
y_oversampled = pd.concat([y_samp, y_min, y_min], axis=0)

# Garbage Collection: Hapus objek besar yang sudah tidak dipakai
import gc
del X_samp, y_samp, X_min, y_min
gc.collect()

# Shuffle data hasil oversampling agar urutan kelas acak
idx = np.random.permutation(len(X_oversampled))
X_final = X_oversampled.iloc[idx].copy()
y_final = y_oversampled.iloc[idx].copy()

# Handling Kategori: Mengisi missing value dan konversi ke string untuk CatBoost
cat_features = X_final.select_dtypes(include=['object', 'category']).columns.tolist()
X_final[cat_features] = X_final[cat_features].fillna("MISSING").astype(str)
X_test_final[cat_features] = X_test_final[cat_features].fillna("MISSING").astype(str)

# Train-Validation Split (85:15)
X_train, X_val, y_train, y_val = train_test_split(X_final, y_final, test_size=0.15, stratify=y_final, random_state=42)

del X_oversampled, y_oversampled
gc.collect()

print(f"Cell 4 Revisi: Ukuran Train: {len(X_train)} baris. RAM terjaga!")

Cell 4 Revisi: Ukuran Train: 1428124 baris. RAM terjaga!


In [ ]:
# Inisialisasi CatBoost: Fokus pada kecepatan dan efisiensi CPU
model = CatBoostClassifier(
    iterations=1200, 
    learning_rate=0.05,
    depth=6, 
    task_type="CPU",
    thread_count=12,
    loss_function='MultiClass',
    eval_metric='Accuracy', 
    early_stopping_rounds=50,
    random_seed=42,
    verbose=100
)

# Menggunakan objek Pool untuk optimasi penggunaan memori CatBoost
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)

# Training dengan Early Stopping untuk mencegah overfit yang lebih parah
model.fit(
    train_pool,
    eval_set=val_pool,
    use_best_model=True
)

0:	learn: 0.5477087	test: 0.5477419	best: 0.5477419 (0)	total: 6.1s	remaining: 2h 1m 58s
100:	learn: 0.8061744	test: 0.9594718	best: 0.9594718 (100)	total: 8m 11s	remaining: 1h 29m 9s
200:	learn: 0.8142696	test: 0.9699193	best: 0.9699193 (200)	total: 17m 33s	remaining: 1h 27m 14s
300:	learn: 0.8173023	test: 0.9734745	best: 0.9734745 (299)	total: 27m 24s	remaining: 1h 21m 51s
400:	learn: 0.8186285	test: 0.9748673	best: 0.9748752 (398)	total: 36m 54s	remaining: 1h 13m 33s
500:	learn: 0.8190206	test: 0.9750498	best: 0.9750498 (492)	total: 46m 15s	remaining: 1h 4m 32s
600:	learn: 0.8192993	test: 0.9752998	best: 0.9753117 (598)	total: 55m 42s	remaining: 55m 31s
700:	learn: 0.8194667	test: 0.9755259	best: 0.9755259 (700)	total: 1h 5m 4s	remaining: 46m 19s
800:	learn: 0.8196004	test: 0.9756450	best: 0.9756450 (794)	total: 1h 14m 20s	remaining: 37m 2s
900:	learn: 0.8197033	test: 0.9756847	best: 0.9756847 (891)	total: 1h 23m 51s	remaining: 27m 49s
1000:	learn: 0.8198210	test: 0.9757759	best: 0.

In [ ]:
# Prediksi pada data validasi
y_val_pred = model.predict(X_val).flatten()

# Output metrik utama kompetisi: Macro F1-Score
print("HASIL EVALUASI AKHIR:")
print(f"Macro F1 Score: {f1_score(y_val, y_val_pred, average='macro'):.4f}")
print("-" * 50)
print(classification_report(y_val, y_val_pred))

HASIL EVALUASI AKHIR:
Macro F1 Score: 0.9784
--------------------------------------------------
                   precision    recall  f1-score   support

 Fraud_Indication       0.99      0.98      0.99      9007
 Navigation_Issue       1.00      0.96      0.98     54121
     Perfect_Trip       0.95      1.00      0.97     98946
 Safety_Violation       0.99      0.95      0.97     36036
Service_Complaint       1.00      0.97      0.98     53912

         accuracy                           0.98    252022
        macro avg       0.99      0.97      0.98    252022
     weighted avg       0.98      0.98      0.98    252022



In [ ]:
# Prediksi pada dataset kompetisi (Test)
test_preds = model.predict(X_test_final).flatten()

# Pembuatan file submission CSV
submission = pd.DataFrame({
    "Trip_ID": df_test["Trip_ID"],
    "Trip_Label": test_preds
})

sub_file = os.path.join(SUB_PATH, "submission_final_3.csv")
submission.to_csv(sub_file, index=False)
print(f"Selesai! Submit file ini: {sub_file}")

Selesai! Submit file ini: ../submissions/submission_final_3.csv


In [ ]:
import os
import joblib

MODEL_DIR = "../models" 

if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

# Simpan modelnya
model_file = os.path.join(MODEL_DIR, "model_akhir_expert.joblib")
joblib.dump(model, model_file, compress=3)

print(f"BERHASIL! Model tersimpan di: {model_file}")

BERHASIL! Model tersimpan di: ../models/model_akhir_expert.csv
